# Session 2 Playground — Notebook to Pipeline

Walk through refactoring Session 1 workflow into an automated batch pipeline.

## 1. Load config and inspect

In [ ]:
import yaml
from pathlib import Path

CONFIG_PATH = "config.example.yaml"
with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

config

## 2. Step-by-step pipeline (live build)

Each cell maps to a function in `pipeline.py`.

In [ ]:
import json
import logging
import sys

import geopandas as gpd

# Import from reference pipeline
sys.path.insert(0, ".")
from pipeline import (
    load_cached_features,
    features_to_gdf,
    process_gdf,
    export_outputs,
    setup_logging,
)

output_dir = Path(config["output"]["dir"])
setup_logging(output_dir)

In [ ]:
# Step 1: Fetch (cached for class)
cached_file = Path(config["api"]["cached_file"])
features = load_cached_features(cached_file)
print(f"Loaded {len(features)} features")

In [ ]:
# Step 2: Features → GeoDataFrame
gdf = features_to_gdf(features)
gdf.head()

In [ ]:
# Step 3: Process
target_crs = config["output"]["target_crs"]
gdf = process_gdf(gdf, target_crs)
gdf[["id", "name", "area_ha", "geometry_valid"]].describe()

In [ ]:
# Step 4: Export
export_outputs(gdf, output_dir)
list(output_dir.glob("*"))

## 3. Run full pipeline from CLI

```bash
cd session-02-automation
cp config.example.yaml config.yaml
python pipeline.py --config config.yaml
```

Expected outputs in `output/`:
- `summary.csv`
- `polygons_processed.gpkg`
- `run.log`

## 4. Beginner exercise

Open `pipeline_template.py` and complete the 3 TODO blocks:
1. Pagination loop in `fetch_all_features()`
2. Processing in `process_gdf()`
3. Export in `export_outputs()`

Reference solution: `pipeline.py`

## 5. n8n hands-on (mandatory)

Follow [`../docs/session02_n8n_hands_on.md`](../docs/session02_n8n_hands_on.md).

Import [`n8n/mapid_polygon_pipeline.json`](n8n/mapid_polygon_pipeline.json), configure credentials, and run workflow to trigger `python pipeline.py`.

**In-class deliverable:** Python pipeline runs + n8n workflow executes successfully.